In [0]:
```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from google.auth.transport.requests import Request
import requests
import uuid

# ============================================================
# PROJECT CONFIGURATION
# Replace placeholders with your environment-specific values.
# Never commit credentials or access tokens to GitHub.
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

# GCS Raw Delta
RAW_PATH = "gs://YOUR_RAW_BUCKET_NAME/customers/"

# BigQuery
BQ_DATASET = "Gcp_Curated"

DQ_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.DATA_QUALITY_RESULTS"
STAGE_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.DIM_CUSTOMER_STAGE"
TARGET_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.DIM_CUSTOMER"

# Temporary GCS bucket used by the Spark BigQuery connector
BQ_TEMP_BUCKET = "gs://YOUR_BQ_TEMP_BUCKET/"

# Watermark table
WATERMARK_TABLE = "YOUR_CATALOG.YOUR_WATERMARK_TABLE"

# Pipeline metadata
PIPELINE_NAME = "customer_curated_pipeline"
SOURCE_SYSTEM = "CloudSQL"

# Generate a unique pipeline run ID
RUN_ID = str(uuid.uuid4())

print("Project:", PROJECT_ID)
print("Raw path:", RAW_PATH)
print("Run ID:", RUN_ID)
```

In [0]:
# ============================================================
# STEP 2 — READ LAST WATERMARK VALUE
# ============================================================

watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "DIM_CUSTOMER")
    .filter(F.col("status") == "SUCCESS")
)

watermark_df.show(truncate=False)

In [0]:
lwv = watermark_df.select("lwv").first()["lwv"]

print("Last Watermark Value:", lwv)

In [0]:
# ============================================================
# STEP 3 — READ RAW DELTA + INCREMENTAL FILTER
# ============================================================

raw_customers = (
    spark.read
    .format("delta")
    .load(RAW_PATH)
)

print("Total raw records:", raw_customers.count())

customer_incremental  = (
    raw_customers
    .filter(F.col("updated_at") > F.lit(lwv))
)

print("Incremental records:", customer_incremental .count())

customer_incremental .show(5, truncate=False)

In [0]:
# ============================================================
# STEP 4 — REMOVE EXACT DUPLICATES
# ============================================================

before_dedup = customer_incremental .count()

customer_incremental= customer_incremental .dropDuplicates()

after_dedup = customer_incremental .count()

print("Records before exact deduplication:", before_dedup)
print("Records after exact deduplication :", after_dedup)
print("Exact duplicates removed           :", before_dedup - after_dedup)

In [0]:
# ============================================================
# STEP 5A — CUSTOMER DQ RULES
# ============================================================

dq_rules = [

    {
        "rule_id": "CUST_001",
        "rule_name": "Customer ID Not Null",
        "rule_type": "NOT_NULL",
        "column_name": "customer_id",
        "condition": F.col("customer_id").isNotNull()
    },

    {
        "rule_id": "CUST_002",
        "rule_name": "Customer ID Positive",
        "rule_type": "POSITIVE",
        "column_name": "customer_id",
        "condition": F.col("customer_id") > 0
    },

    {
        "rule_id": "CUST_003",
        "rule_name": "Valid Email",
        "rule_type": "FORMAT",
        "column_name": "email",
        "condition": (
            F.col("email").isNull()
            | F.col("email").rlike(
                r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            )
        )
    },

    {
        "rule_id": "CUST_004",
        "rule_name": "Created At Not Null",
        "rule_type": "NOT_NULL",
        "column_name": "created_at",
        "condition": F.col("created_at").isNotNull()
    },

    {
        "rule_id": "CUST_005",
        "rule_name": "Updated At Not Null",
        "rule_type": "NOT_NULL",
        "column_name": "updated_at",
        "condition": F.col("updated_at").isNotNull()
    },

    {
        "rule_id": "CUST_006",
        "rule_name": "Updated At Greater Than Or Equal To Created At",
        "rule_type": "DATE_CHECK",
        "column_name": "updated_at",
        "condition": F.col("updated_at") >= F.col("created_at")
    }
]

print("Number of DQ rules:", len(dq_rules))

In [0]:
# ============================================================
# STEP 5B — EXECUTE DQ RULES
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

dq_results = []

total_records = customer_incremental .count()

for rule in dq_rules:

    passed_records = (
        customer_incremental 
        .filter(rule["condition"])
        .count()
    )

    failed_records = total_records - passed_records

    if total_records == 0:
        pass_percentage = 100.0
        fail_percentage = 100.0
    else:
        pass_percentage = passed_records * 100.0 / total_records
        fail_percentage = failed_records * 100.0 / total_records

    dq_status = "PASS" if failed_records == 0 else "FAIL"

    dq_results.append((
        RUN_ID,
        "DIM_CUSTOMER",
        rule["column_name"],
        rule["rule_id"],
        rule["rule_name"],
        rule["rule_type"],
        "RECORD",
        total_records,
        passed_records,
        failed_records,
        round(pass_percentage, 2),
        round(fail_percentage, 2),
        dq_status,
        None,
        SOURCE_SYSTEM,
        PIPELINE_NAME
    ))


# ------------------------------------------------------------
# Explicit schema
# ------------------------------------------------------------

dq_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("dq_rule_id", StringType(), True),
    StructField("dq_rule_name", StringType(), True),
    StructField("dq_rule_type", StringType(), True),
    StructField("check_level", StringType(), True),
    StructField("total_records", LongType(), True),
    StructField("passed_records", LongType(), True),
    StructField("failed_records", LongType(), True),
    StructField("pass_percentage", DoubleType(), True),
    StructField("fail_percentage", DoubleType(), True),
    StructField("dq_status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("pipeline_name", StringType(), True)
])


dq_results_df = spark.createDataFrame(
    dq_results,
    schema=dq_schema
)

dq_results_df.show(truncate=False)

In [0]:
# ============================================================
# STEP 5C — FINALIZE DQ RESULT DATAFRAME
# ============================================================

dq_results_final = (
    dq_results_df

    # Unique key for each DQ result
    .withColumn(
        "dq_result_sk",
        F.monotonically_increasing_id()
    )

    # Execution timestamp
    .withColumn(
        "execution_timestamp",
        F.current_timestamp()
    )

    # Data warehouse created timestamp
    .withColumn(
        "dw_created_at",
        F.current_timestamp()
    )

    # Convert percentages to the BigQuery NUMERIC(5,2) type
    .withColumn(
        "pass_percentage",
        F.round(F.col("pass_percentage"), 2).cast("decimal(5,2)")
    )
    .withColumn(
        "fail_percentage",
        F.round(F.col("fail_percentage"), 2).cast("decimal(5,2)")
    )
)

dq_results_final = dq_results_final.select(
    "dq_result_sk",
    "run_id",
    "table_name",
    "column_name",
    "dq_rule_id",
    "dq_rule_name",
    "dq_rule_type",
    "check_level",
    "total_records",
    "passed_records",
    "failed_records",
    "pass_percentage",
    "fail_percentage",
    "dq_status",
    "error_message",
    "source_system",
    "pipeline_name",
    "execution_timestamp",
    "dw_created_at"
)

dq_results_final.show(truncate=False)

In [0]:
# ============================================================
# STEP 6 — WRITE DQ RESULTS TO BIGQUERY
# ============================================================

(
    dq_results_final
    .write
    .format("bigquery")
    .option(
        "table",
        DQ_TABLE
    )
    .option(
        "temporaryGcsBucket",
        BQ_TEMP_BUCKET
    )
    .mode("append")
    .save()
)

print("DQ results written successfully to:", DQ_TABLE)

In [0]:
# ============================================================
# STEP 6.1 — PREPARE CUSTOMER DATA FOR SCD2
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

scd2_source = (
    customer_incremental

    # Business key
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "state",
        "country",
        "customer_segment",
        "created_at",
        "updated_at"
    )

    # Create hash from business attributes
    # Technical timestamps are intentionally excluded
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("first_name"), F.lit("")),
                F.coalesce(F.col("last_name"), F.lit("")),
                F.coalesce(F.col("email"), F.lit("")),
                F.coalesce(F.col("phone"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("state"), F.lit("")),
                F.coalesce(F.col("country"), F.lit("")),
                F.coalesce(F.col("customer_segment"), F.lit(""))
            ),
            256
        )
    )

    # SCD2 effective start
    .withColumn(
        "effective_from",
        F.col("updated_at")
    )
)

print("SCD2 source records:", scd2_source.count())

display(scd2_source)

In [0]:
# ============================================================
# STEP 6.2 — CREATE SCD2 RECORD HASH
# ============================================================

from pyspark.sql import functions as F

scd2_source = (
    customer_incremental

    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "state",
        "country",
        "customer_segment",
        "created_at",
        "updated_at"
    )

    # Hash business/descriptive columns
    # Technical columns are NOT included
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("first_name"), F.lit("")),
                F.coalesce(F.col("last_name"), F.lit("")),
                F.coalesce(F.col("email"), F.lit("")),
                F.coalesce(F.col("phone"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("state"), F.lit("")),
                F.coalesce(F.col("country"), F.lit("")),
                F.coalesce(F.col("customer_segment"), F.lit(""))
            ),
            256
        )
    )

    # SCD2 version starts when source record was updated
    .withColumn(
        "effective_from",
        F.col("updated_at")
    )
)

print("SCD2 source records:", scd2_source.count())

display(scd2_source)

In [0]:
# ============================================================
# STEP 6.3 — REMOVE CONSECUTIVE UNCHANGED VERSIONS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Order each customer's records chronologically
change_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("effective_from").asc(),
        F.col("updated_at").asc()
    )
)

# Compare current hash with previous version
scd2_source_changed = (
    scd2_source

    .withColumn(
        "previous_hash",
        F.lag("record_hash").over(change_window)
    )

    # Keep:
    # 1. First record for the customer
    # 2. Record where business attributes changed
    .filter(
        F.col("previous_hash").isNull() |
        (F.col("record_hash") != F.col("previous_hash"))
    )

    .drop("previous_hash")
)

print(
    "Records before removing unchanged versions:",
    scd2_source.count()
)

print(
    "Records after removing unchanged versions:",
    scd2_source_changed.count()
)

print(
    "Unchanged versions removed:",
    scd2_source.count() - scd2_source_changed.count()
)

display(
    scd2_source_changed.orderBy(
        "customer_id",
        "effective_from"
    )
)

In [0]:
# ============================================================
# STEP 6.4 — CREATE SCD2 VERSION + EFFECTIVE DATES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Window for chronological ordering
version_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("effective_from").asc(),
        F.col("updated_at").asc()
    )
)

# Assign version number
scd2_source_ordered = (
    scd2_source_changed
    .withColumn(
        "version_number",
        F.row_number().over(version_window)
    )
)

# Window for finding the next version
lead_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("effective_from").asc(),
        F.col("updated_at").asc()
    )
)

# Calculate effective_to and current flag
scd2_ready = (
    scd2_source_ordered

    .withColumn(
        "next_effective_from",
        F.lead("effective_from").over(lead_window)
    )

    .withColumn(
        "effective_to",
        F.coalesce(
            F.col("next_effective_from"),
            F.to_timestamp(
                F.lit("9999-12-31 23:59:59")
            )
        )
    )

    .withColumn(
        "is_current",
        F.col("next_effective_from").isNull()
    )

    .drop("next_effective_from")
)

print("SCD2 records:", scd2_ready.count())

print("Current records:")
print(
    scd2_ready
    .filter(F.col("is_current") == True)
    .count()
)

display(
    scd2_ready
    .orderBy("customer_id", "version_number")
)

In [0]:
# ============================================================
# STEP 6.5 — ADD WAREHOUSE TECHNICAL COLUMNS
# ============================================================

from pyspark.sql import functions as F

scd2_ready = (
    scd2_ready

    # --------------------------------------------------------
    # Surrogate key
    # --------------------------------------------------------
    # Deterministic key based on:
    # customer_id + effective_from
    #
    # Same customer version will always get the same key.
    # --------------------------------------------------------
    .withColumn(
        "customer_sk",
        (
            F.pmod(
                F.xxhash64(
                    F.col("customer_id"),
                    F.col("effective_from")
                ),
                F.lit(9223372036854775806)
            ) + F.lit(1)
        ).cast("long")
    )

    # --------------------------------------------------------
    # Data warehouse audit timestamps
    # --------------------------------------------------------
    .withColumn(
        "dw_created_at",
        F.current_timestamp()
    )

    .withColumn(
        "dw_updated_at",
        F.current_timestamp()
    )
)

# ------------------------------------------------------------
# Arrange columns exactly like BigQuery target
# ------------------------------------------------------------

scd2_ready = scd2_ready.select(
    "customer_sk",
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "state",
    "country",
    "customer_segment",
    "created_at",
    "updated_at",
    "effective_from",
    "effective_to",
    "is_current",
    "version_number",
    "record_hash",
    "dw_created_at",
    "dw_updated_at"
)

print("Final SCD2 stage records:", scd2_ready.count())

display(
    scd2_ready
    .orderBy("customer_id", "version_number")
)

In [0]:
scd2_ready.select(
    "customer_sk",
    "customer_id",
    "version_number",
    "effective_from",
    "effective_to",
    "is_current",
    "record_hash"
).orderBy(
    "customer_id",
    "version_number"
).show(20, truncate=False)

In [0]:
# ============================================================
# STEP 6.6 — WRITE SCD2 DATA TO BIGQUERY STAGE
# ============================================================

STAGE_TABLE = (
    "gcp-data-engineering-504911."
    "Gcp_Curated.DIM_CUSTOMER_STAGE"
)

(
    scd2_ready
    .write
    .format("bigquery")
    .mode("overwrite")
    .option(
        "project",
        "gcp-data-engineering-504911"
    )
    .option(
        "parentProject",
        "gcp-data-engineering-504911"
    )
    .option(
        "table",
        STAGE_TABLE
    )
    .option(
        "temporaryGcsBucket",
        "bq-databricks-temp-504911"
    )
    .save()
)

print("SCD2 stage table written successfully.")

In [0]:
# ============================================================
# VERIFY STAGE TABLE
# ============================================================

stage_check = (
    spark.read
    .format("bigquery")
    .option(
        "table",
        STAGE_TABLE
    )
    .load()
)

print("Stage record count:", stage_check.count())

stage_check.select(
    "customer_id",
    "version_number",
    "effective_from",
    "effective_to",
    "is_current"
).orderBy(
    "customer_id",
    "version_number"
).show(103, truncate=False)

In [0]:
# ============================================================
# STEP 6.7 — SINGLE SCD2 MERGE
# Handles BOTH:
#   1. Initial load
#   2. Incremental load
# ============================================================

import requests
from google.auth.transport.requests import Request

PROJECT_ID = "gcp-data-engineering-504911"

TARGET_TABLE = (
    "`gcp-data-engineering-504911."
    "Gcp_Curated.DIM_CUSTOMER`"
)

STAGE_TABLE = (
    "`gcp-data-engineering-504911."
    "Gcp_Curated.DIM_CUSTOMER_STAGE`"
)

# ------------------------------------------------------------
# SCD2 MERGE
#
# Each stage record is represented twice:
#
# 1. expire row
#    merge_key = customer_id
#    -> matches current target record
#    -> expires old version
#
# 2. insert row
#    merge_key = NULL
#    -> cannot match target
#    -> inserts new version
#
# This allows one MERGE to perform both UPDATE + INSERT.
# ------------------------------------------------------------

merge_query = f"""
MERGE {TARGET_TABLE} AS T

USING
(
    -- ========================================================
    -- ROW 1: EXPIRE CURRENT RECORD
    -- ========================================================
    SELECT
        customer_id AS merge_customer_id,
        customer_sk,
        customer_id,
        first_name,
        last_name,
        email,
        phone,
        city,
        state,
        country,
        customer_segment,
        created_at,
        updated_at,
        effective_from,
        effective_to,
        is_current,
        version_number,
        record_hash,
        dw_created_at,
        dw_updated_at,
        'EXPIRE' AS merge_action

    FROM {STAGE_TABLE}

    UNION ALL

    -- ========================================================
    -- ROW 2: INSERT NEW VERSION
    -- NULL merge key guarantees NOT MATCHED
    -- ========================================================
    SELECT
        CAST(NULL AS INT64) AS merge_customer_id,
        customer_sk,
        customer_id,
        first_name,
        last_name,
        email,
        phone,
        city,
        state,
        country,
        customer_segment,
        created_at,
        updated_at,
        effective_from,
        effective_to,
        is_current,
        version_number,
        record_hash,
        dw_created_at,
        dw_updated_at,
        'INSERT' AS merge_action

    FROM {STAGE_TABLE}

) AS S

ON T.customer_id = S.merge_customer_id
AND T.is_current = TRUE

-- ============================================================
-- EXPIRE OLD CURRENT VERSION
-- ============================================================

WHEN MATCHED
AND S.merge_action = 'EXPIRE'
AND T.record_hash != S.record_hash

THEN UPDATE SET
    effective_to = S.effective_from,
    is_current = FALSE,
    dw_updated_at = CURRENT_TIMESTAMP()

-- ============================================================
-- INSERT NEW VERSION
--
-- This executes for:
--   - New customer
--   - Changed customer
--   - Initial load
-- ============================================================

WHEN NOT MATCHED
AND S.merge_action = 'INSERT'

THEN INSERT
(
    customer_sk,
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    city,
    state,
    country,
    customer_segment,
    created_at,
    updated_at,
    effective_from,
    effective_to,
    is_current,
    version_number,
    record_hash,
    dw_created_at,
    dw_updated_at
)

VALUES
(
    S.customer_sk,
    S.customer_id,
    S.first_name,
    S.last_name,
    S.email,
    S.phone,
    S.city,
    S.state,
    S.country,
    S.customer_segment,
    S.created_at,
    S.updated_at,
    S.effective_from,
    S.effective_to,
    S.is_current,
    S.version_number,
    S.record_hash,
    S.dw_created_at,
    S.dw_updated_at
)
"""

# ------------------------------------------------------------
# Get Databricks → GCP service credential
# ------------------------------------------------------------

provider = dbutils.credentials.getServiceCredentialsProvider(
    "bigquery"
)

provider.refresh(Request())

# ------------------------------------------------------------
# BigQuery REST API
# ------------------------------------------------------------

url = (
    "https://bigquery.googleapis.com/bigquery/v2/"
    f"projects/{PROJECT_ID}/queries"
)

headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

body = {
    "query": merge_query,
    "useLegacySql": False,
    "location": "asia-south2"
}

response = requests.post(
    url,
    headers=headers,
    json=body
)

print("HTTP status:", response.status_code)
print(response.text[:5000])

if response.status_code != 200:
    raise Exception(
        f"BigQuery MERGE failed: {response.text}"
    )

print("SCD2 MERGE completed successfully.")

In [0]:
# ============================================================
# STEP 6.9 — VERIFY DIM_CUSTOMER SCD2 RESULT
# ============================================================

verify_query = """
SELECT
    COUNT(*) AS total_records,
    COUNTIF(is_current = TRUE) AS current_records,
    COUNTIF(is_current = FALSE) AS historical_records,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    COUNT(DISTINCT customer_sk) AS distinct_surrogate_keys
FROM `gcp-data-engineering-504911.Gcp_Curated.DIM_CUSTOMER`
"""

response = requests.post(
    url,
    headers=headers,
    json={
        "query": verify_query,
        "useLegacySql": False,
        "location": "asia-south2"
    }
)

print("HTTP status:", response.status_code)
print(response.text)

In [0]:
# ============================================================
# STEP 6.10 — FIND CUSTOMERS WITH NO CURRENT VERSION
# ============================================================

missing_current_query = """
SELECT
    customer_id,
    COUNT(*) AS total_versions,
    COUNTIF(is_current = TRUE) AS current_versions,
    COUNTIF(is_current = FALSE) AS historical_versions
FROM `gcp-data-engineering-504911.Gcp_Curated.DIM_CUSTOMER`
GROUP BY customer_id
HAVING COUNTIF(is_current = TRUE) = 0
ORDER BY customer_id
"""

response = requests.post(
    url,
    headers=headers,
    json={
        "query": missing_current_query,
        "useLegacySql": False,
        "location": "asia-south2"
    }
)

print("HTTP status:", response.status_code)
print(response.text)

In [0]:
# ============================================================
# STEP 6.8 — UPDATE WATERMARK AFTER SUCCESSFUL MERGE
# ============================================================

if total_records == 0:
    new_lwv = lwv
    print("No new records. Keeping previous LWV:", new_lwv)
else:
    new_lwv = (
        scd2_ready
        .agg(F.max("updated_at").alias("max_updated_at"))
        .first()["max_updated_at"]
    )

    print("New LWV:", new_lwv)

In [0]:
# Update DIM_CUSTOMER watermark to the latest successfully processed timestamp

    
spark.sql(f"""
UPDATE {WATERMARK_TABLE}
SET
    lwv = TIMESTAMP('{new_lwv}'),
    status = 'SUCCESS'
WHERE table_name = 'DIM_CUSTOMER'
""")

print("DIM_CUSTOMER watermark updated successfully.")

In [0]:
spark.table(WATERMARK_TABLE) \
    .filter(F.col("table_name") == "DIM_CUSTOMER") \
    .show(truncate=False)